# pLDDT Confidence Analysis

Analyze AlphaFold3 confidence scores (pLDDT) across:
- Overall distribution
- Per-domain type
- Linkers vs core domains
- Implications for training strategy

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from src.data.cif_parser import CIFParser
from src.data.annotation_parser import AnnotationParser, CORE_DOMAINS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load pre-computed statistics if available
stats_file = Path('../data/processed/plddt_statistics.json')

if stats_file.exists():
    with open(stats_file) as f:
        plddt_stats = json.load(f)
    print('Loaded pre-computed pLDDT statistics')
    print(f"  Mean pLDDT: {plddt_stats['mean']:.1f}")
    print(f"  N structures: {plddt_stats['n_structures']}")
else:
    print('No pre-computed statistics. Run 01_prepare_data.py first.')

## Load Sample Structures for Detailed Analysis

In [ ]:
# Load annotations
annotation_csv = Path('../fragments_for_prediction_COREONLY.csv')
annotation_parser = AnnotationParser(annotation_csv)

# Load a sample of structures for detailed pLDDT analysis
cif_dir = Path('../data/raw')  # Adjust path as needed

if cif_dir.exists():
    cif_files = list(cif_dir.glob('*.cif'))[:100]  # Sample 100 structures
    print(f'Found {len(cif_files)} CIF files for analysis')
else:
    print(f'CIF directory not found: {cif_dir}')
    cif_files = []

In [ ]:
# Parse sample structures
if cif_files:
    parser = CIFParser()
    
    all_plddt = []
    domain_plddt = {dt: [] for dt in list(CORE_DOMAINS) + ['linker']}
    
    for cif_path in tqdm(cif_files, desc='Parsing structures'):
        try:
            structure = parser.parse(cif_path)
            all_plddt.extend(structure.plddt.tolist())
            
            # Get matching annotation
            fragment_id = cif_path.stem
            # Try to match without model suffix
            import re
            base_id = re.sub(r'_model_\d+$', '', fragment_id)
            
            ann = annotation_parser.get(base_id) or annotation_parser.get(fragment_id)
            
            if ann:
                for domain in ann.domains:
                    indices = domain.get_residue_indices(zero_indexed=True)
                    indices = indices[indices < len(structure.plddt)]
                    
                    if domain.is_linker:
                        domain_plddt['linker'].extend(structure.plddt[indices].tolist())
                    elif domain.domain_type in domain_plddt:
                        domain_plddt[domain.domain_type].extend(structure.plddt[indices].tolist())
        except Exception as e:
            continue
    
    print(f'Collected {len(all_plddt)} pLDDT values')

## Overall pLDDT Distribution

In [ ]:
if all_plddt:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax = axes[0]
    ax.hist(all_plddt, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(70, color='red', linestyle='--', linewidth=2, label='High threshold (70)')
    ax.axvline(50, color='orange', linestyle='--', linewidth=2, label='Low threshold (50)')
    ax.set_xlabel('pLDDT Score')
    ax.set_ylabel('Count')
    ax.set_title('Overall pLDDT Distribution')
    ax.legend()
    
    # Cumulative
    ax = axes[1]
    sorted_plddt = np.sort(all_plddt)
    cumulative = np.arange(1, len(sorted_plddt) + 1) / len(sorted_plddt)
    ax.plot(sorted_plddt, cumulative)
    ax.axvline(70, color='red', linestyle='--', label='70')
    ax.axvline(50, color='orange', linestyle='--', label='50')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('pLDDT Score')
    ax.set_ylabel('Cumulative Fraction')
    ax.set_title('Cumulative pLDDT Distribution')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    plddt_array = np.array(all_plddt)
    print(f'pLDDT Statistics:')
    print(f'  >90 (very high):   {(plddt_array > 90).mean()*100:.1f}%')
    print(f'  70-90 (confident): {((plddt_array >= 70) & (plddt_array <= 90)).mean()*100:.1f}%')
    print(f'  50-70 (low):       {((plddt_array >= 50) & (plddt_array < 70)).mean()*100:.1f}%')
    print(f'  <50 (very low):    {(plddt_array < 50).mean()*100:.1f}%')

## pLDDT by Domain Type

In [ ]:
if domain_plddt:
    # Filter domains with data
    domains_with_data = {k: v for k, v in domain_plddt.items() if len(v) > 100}
    
    if domains_with_data:
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Sort by median pLDDT
        sorted_domains = sorted(domains_with_data.items(), 
                                key=lambda x: np.median(x[1]), reverse=True)
        
        data = [v for k, v in sorted_domains]
        labels = [k for k, v in sorted_domains]
        
        bp = ax.boxplot(data, labels=labels, patch_artist=True)
        
        # Color core domains vs linkers
        for i, (patch, label) in enumerate(zip(bp['boxes'], labels)):
            if label == 'linker':
                patch.set_facecolor('coral')
            else:
                patch.set_facecolor('steelblue')
        
        ax.axhline(70, color='red', linestyle='--', alpha=0.7, label='High threshold')
        ax.axhline(50, color='orange', linestyle='--', alpha=0.7, label='Low threshold')
        ax.set_xlabel('Domain Type')
        ax.set_ylabel('pLDDT Score')
        ax.set_title('pLDDT Distribution by Domain Type')
        ax.legend()
        ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Print median pLDDT per domain
        print('\nMedian pLDDT by domain type:')
        for domain, values in sorted_domains:
            print(f'  {domain:8s}: {np.median(values):.1f} (n={len(values):,})')

## Implications for Training

Based on the pLDDT analysis:

1. **High-confidence residues (pLDDT > 70)**: These should contribute fully to training loss
2. **Medium-confidence residues (50-70)**: Include in graph for context but mask from loss
3. **Low-confidence residues (< 50)**: Exclude entirely - likely "spaghetti" predictions

This matches the recommendations from both Gemini and GPT.